In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Charger le dataset original
df = pd.read_csv("../../data/localnet_dynamics.csv")

# Afficher les dimensions
print("Dimensions du dataset :", df.shape)

# Afficher les premières lignes
display(df.head())

Dimensions du dataset : (54617, 67)


C:\Users\dell\AppData\Local\Temp\ipykernel_32960\4128352036.py:6: DtypeWarning: Columns (47: wifi_phy) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/localnet_dynamics.csv")


,run_id,host_label,sample_index,timestamp_utc,timestamp_local,epoch_s,elapsed_s,link_type,gateway_ip,dns_server,...,sample_duration_s,label_outage,label_packet_loss,label_high_latency,label_high_jitter,label_gateway_degraded,label_anomaly,event_category,event_description,notes
0,desktop-eth_20260920_123407,desktop-eth,0,2026-09-20T16:34:22Z,2026-09-20T12:34:22-04:00,1789922062,0.0,unknown,192.168.1.1,192.168.1.1,...,65.70,0,0,0,0,0,0,NaN,NaN,NaN
1,desktop-eth_20260920_123407,desktop-eth,1,2026-09-20T16:35:52Z,2026-09-20T12:35:52-04:00,1789922152,90.0,unknown,192.168.1.1,192.168.1.1,...,12.41,0,1,0,0,0,1,NaN,NaN,NaN
2,desktop-eth_20260920_123407,desktop-eth,2,2026-09-20T16:36:22Z,2026-09-20T12:36:22-04:00,1789922182,120.0,unknown,192.168.1.1,192.168.1.1,...,10.20,0,0,0,0,0,0,NaN,NaN,NaN
3,desktop-eth_20260920_123407,desktop-eth,3,2026-09-20T16:36:52Z,2026-09-20T12:36:52-04:00,1789922212,150.0,unknown,192.168.1.1,192.168.1.1,...,9.32,0,0,0,0,0,0,NaN,NaN,NaN
4,desktop-eth_20260920_123407,desktop-eth,4,2026-09-20T16:37:22Z,2026-09-20T12:37:22-04:00,1789922242,180.0,unknown,192.168.1.1,192.168.1.1,...,11.07,0,0,1,1,0,1,NaN,NaN,NaN


In [2]:
# Créer une copie de travail
df_work = df.copy()

# Vérifier les dimensions
print("Dimensions de la copie :", df_work.shape)

# Vérifier les types de données
print("\nTypes des colonnes :")
display(df_work.dtypes)

# Vérifier les valeurs manquantes
print("\nNombre de valeurs manquantes :")
display(
    df_work.isna().sum()
    .sort_values(ascending=False)
    .head(20)
)

Dimensions de la copie : (54617, 67)

Types des colonnes :


run_id                        str
host_label                    str
sample_index                int64
timestamp_utc                 str
timestamp_local               str
                           ...   
label_gateway_degraded      int64
label_anomaly               int64
event_category            float64
event_description         float64
notes                         str
Length: 67, dtype: object


Nombre de valeurs manquantes :


download_loaded_latency_ms    54617
upload_loaded_latency_ms      54617
event_category                54617
lan_retransmits               54617
event_description             54617
wifi_ssid                     54617
lan_throughput_mbps           54617
wifi_phy                      54613
wifi_rssi_dbm                 54613
wifi_rx_mbps                  54613
wifi_signal_pct               54613
wifi_channel                  54613
wifi_band_ghz                 54613
wifi_tx_mbps                  54613
notes                         54558
speedtest_ping_ms             54236
speedtest_server              54157
upload_mbps                   54157
download_mbps                 54157
speedtest_server_id           54157
dtype: int64

In [3]:
# Colonnes utiles pour notre étude
colonnes_a_conserver = [
    "timestamp_utc",
    "hour_local",
    "weekday_local",
    "is_weekend",
    "rtt_avg_ms",
    "rtt_min_ms",
    "rtt_max_ms",
    "rtt_median_ms",
    "rtt_stddev_ms",
    "jitter_ms",
    "packet_loss_pct",
    "gateway_rtt_avg_ms",
    "gateway_jitter_ms",
    "gateway_packet_loss_pct",
    "tcp_connect_median_ms",
    "dns_resolve_ms",
    "dns_recursive_ms",
    "http_ms",
    "gap_before_s"
]

# Créer notre dataset de travail réduit
df_work = df_work[colonnes_a_conserver].copy()

# Convertir la date en véritable format datetime
df_work["timestamp_utc"] = pd.to_datetime(
    df_work["timestamp_utc"],
    utc=True
)

# Trier chronologiquement
df_work = df_work.sort_values(
    "timestamp_utc"
).reset_index(drop=True)

# Vérifications
print("Dimensions après sélection :", df_work.shape)

print("\nColonnes conservées :")
print(df_work.columns.tolist())

print("\nValeurs manquantes :")
display(df_work.isna().sum().sort_values(ascending=False))

print("\nPremières observations :")
display(df_work.head())

Dimensions après sélection : (54617, 19)

Colonnes conservées :
['timestamp_utc', 'hour_local', 'weekday_local', 'is_weekend', 'rtt_avg_ms', 'rtt_min_ms', 'rtt_max_ms', 'rtt_median_ms', 'rtt_stddev_ms', 'jitter_ms', 'packet_loss_pct', 'gateway_rtt_avg_ms', 'gateway_jitter_ms', 'gateway_packet_loss_pct', 'tcp_connect_median_ms', 'dns_resolve_ms', 'dns_recursive_ms', 'http_ms', 'gap_before_s']

Valeurs manquantes :


http_ms                    52
dns_recursive_ms           27
dns_resolve_ms             22
tcp_connect_median_ms      19
gateway_rtt_avg_ms         18
rtt_stddev_ms              18
gateway_jitter_ms          18
rtt_avg_ms                 18
rtt_min_ms                 18
rtt_max_ms                 18
rtt_median_ms              18
jitter_ms                  18
gap_before_s                1
timestamp_utc               0
hour_local                  0
weekday_local               0
is_weekend                  0
packet_loss_pct             0
gateway_packet_loss_pct     0
dtype: int64


Premières observations :


,timestamp_utc,hour_local,weekday_local,is_weekend,rtt_avg_ms,rtt_min_ms,rtt_max_ms,rtt_median_ms,rtt_stddev_ms,jitter_ms,packet_loss_pct,gateway_rtt_avg_ms,gateway_jitter_ms,gateway_packet_loss_pct,tcp_connect_median_ms,dns_resolve_ms,dns_recursive_ms,http_ms,gap_before_s
0,2026-09-20 16:34:22+00:00,12,6,1,10.500,10.0,11.0,10.5,0.500,0.667,0.0,0.0,0.0,0.0,12.005,26.420,273.373,430.493,NaN
1,2026-09-20 16:35:52+00:00,12,6,1,11.222,10.0,12.0,11.0,0.786,0.500,10.0,0.0,0.0,0.0,12.208,132.626,133.356,129.852,90.0
2,2026-09-20 16:36:22+00:00,12,6,1,10.900,10.0,13.0,11.0,0.943,0.889,0.0,0.4,0.5,0.0,13.766,445.539,97.841,130.988,30.0
3,2026-09-20 16:36:52+00:00,12,6,1,10.500,10.0,13.0,10.0,0.922,0.889,0.0,0.2,0.5,0.0,28.913,140.461,66.917,1383.776,30.0
4,2026-09-20 16:37:22+00:00,12,6,1,36.300,10.0,270.0,10.0,77.903,58.000,0.0,0.4,0.5,0.0,13.767,32.678,378.545,331.481,30.0


In [4]:
# Identifier les lignes où la latence moyenne est manquante
masque_rtt = df_work["rtt_avg_ms"].isna()

# Récupérer les informations correspondantes depuis le dataset original
colonnes_controle = [
    "timestamp_utc",
    "label_outage",
    "packet_loss_pct",
    "packets_sent",
    "packets_received",
    "label_anomaly"
]

lignes_sans_rtt = df.loc[masque_rtt, colonnes_controle]

print("Nombre de lignes sans RTT :", masque_rtt.sum())

print("\nRépartition des pannes :")
print(lignes_sans_rtt["label_outage"].value_counts())

print("\nRépartition des pertes de paquets :")
print(lignes_sans_rtt["packet_loss_pct"].value_counts())

print("\nDétail des observations :")
display(lignes_sans_rtt)

Nombre de lignes sans RTT : 18

Répartition des pannes :
label_outage
1    18
Name: count, dtype: int64

Répartition des pertes de paquets :
packet_loss_pct
100.0    18
Name: count, dtype: int64

Détail des observations :


,timestamp_utc,label_outage,packet_loss_pct,packets_sent,packets_received,label_anomaly
2884,2026-09-21T17:03:22Z,1,100.0,10,0,1
2885,2026-09-21T17:03:52Z,1,100.0,10,0,1
2886,2026-09-21T17:04:22Z,1,100.0,10,0,1
2887,2026-09-21T17:04:54Z,1,100.0,10,0,1
10750,2026-09-24T11:10:56Z,1,100.0,10,0,1
10751,2026-09-24T11:11:26Z,1,100.0,10,0,1
10752,2026-09-24T11:11:56Z,1,100.0,10,0,1
10753,2026-09-24T11:12:28Z,1,100.0,10,0,1
10754,2026-09-24T11:12:56Z,1,100.0,10,0,1
17169,2026-09-26T17:11:29Z,1,100.0,10,0,1


In [5]:
# Calculer les intervalles entre les observations
intervalles = df_work["gap_before_s"]

print("Statistiques des intervalles :")
display(intervalles.describe())

print("\nNombre d'intervalles par catégorie :")

print("Entre 25 et 35 secondes :",
      intervalles.between(25, 35).sum())

print("Entre 36 et 45 secondes :",
      intervalles.between(36, 45).sum())

print("Entre 46 et 75 secondes :",
      intervalles.between(46, 75).sum())

print("Supérieurs à 75 secondes :",
      (intervalles > 75).sum())

print("\nNombre de valeurs manquantes :",
      intervalles.isna().sum())

Statistiques des intervalles :


count    54616.000000
mean        30.271734
std          3.272058
min         10.000000
25%         30.000000
50%         30.000000
75%         30.000000
max        122.000000
Name: gap_before_s, dtype: float64


Nombre d'intervalles par catégorie :
Entre 25 et 35 secondes : 53889
Entre 36 et 45 secondes : 124
Entre 46 et 75 secondes : 420
Supérieurs à 75 secondes : 39

Nombre de valeurs manquantes : 1


In [6]:
# Identifier les intervalles supérieurs à 45 secondes
grands_intervalles = df_work.loc[
    df_work["gap_before_s"] > 45,
    ["timestamp_utc", "gap_before_s", "rtt_avg_ms", "packet_loss_pct"]
].copy()

print("Nombre d'intervalles supérieurs à 45 secondes :",
      len(grands_intervalles))

print("\nRépartition des grands intervalles :")
display(
    grands_intervalles["gap_before_s"]
    .describe()
)

print("\nLes 15 plus grands intervalles :")
display(
    grands_intervalles
    .sort_values("gap_before_s", ascending=False)
    .head(15)
)

Nombre d'intervalles supérieurs à 45 secondes : 459

Répartition des grands intervalles :


count    459.000000
mean      62.965142
std       10.272642
min       46.000000
25%       60.000000
50%       60.000000
75%       60.000000
max      122.000000
Name: gap_before_s, dtype: float64


Les 15 plus grands intervalles :


,timestamp_utc,gap_before_s,rtt_avg_ms,packet_loss_pct
46140,2026-10-06 20:36:41+00:00,122.0,10.900,0.0
46257,2026-10-06 21:36:39+00:00,120.0,12.100,0.0
46816,2026-10-07 02:20:09+00:00,120.0,10.400,0.0
39828,2026-10-04 15:36:37+00:00,119.0,10.500,0.0
43953,2026-10-06 02:15:09+00:00,118.0,10.600,0.0
23046,2026-09-28 18:36:06+00:00,95.0,14.222,10.0
49701,2026-10-08 02:36:14+00:00,93.0,10.500,0.0
14845,2026-09-25 21:35:59+00:00,92.0,10.700,0.0
23640,2026-09-28 23:36:01+00:00,90.0,16.444,10.0
33161,2026-10-02 07:36:05+00:00,90.0,12.000,0.0


In [7]:
# Créer une copie dédiée à la modélisation
df_model = df_work.copy()

# Créer la cible : latence du prochain échantillon
df_model["rtt_future"] = df_model["rtt_avg_ms"].shift(-1)

# Récupérer l'intervalle entre l'observation actuelle et la prochaine
df_model["gap_future_s"] = df_model["gap_before_s"].shift(-1)

print("Dimensions :", df_model.shape)

print("\nPremières observations :")
display(
    df_model[
        ["timestamp_utc", "rtt_avg_ms",
         "rtt_future", "gap_future_s"]
    ].head(10)
)

print("\nDernières observations :")
display(
    df_model[
        ["timestamp_utc", "rtt_avg_ms",
         "rtt_future", "gap_future_s"]
    ].tail()
)

Dimensions : (54617, 21)

Premières observations :


,timestamp_utc,rtt_avg_ms,rtt_future,gap_future_s
0,2026-09-20 16:34:22+00:00,10.500,11.222,90.0
1,2026-09-20 16:35:52+00:00,11.222,10.900,30.0
2,2026-09-20 16:36:22+00:00,10.900,10.500,30.0
3,2026-09-20 16:36:52+00:00,10.500,36.300,30.0
4,2026-09-20 16:37:22+00:00,36.300,10.800,30.0
5,2026-09-20 16:37:52+00:00,10.800,12.400,30.0
6,2026-09-20 16:38:22+00:00,12.400,11.000,30.0
7,2026-09-20 16:38:52+00:00,11.000,11.000,30.0
8,2026-09-20 16:39:22+00:00,11.000,12.400,30.0
9,2026-09-20 16:39:52+00:00,12.400,10.300,30.0



Dernières observations :


,timestamp_utc,rtt_avg_ms,rtt_future,gap_future_s
54612,2026-10-09 19:47:43+00:00,10.9,10.4,30.0
54613,2026-10-09 19:48:13+00:00,10.4,10.4,30.0
54614,2026-10-09 19:48:43+00:00,10.4,12.1,30.0
54615,2026-10-09 19:49:13+00:00,12.1,10.3,30.0
54616,2026-10-09 19:49:43+00:00,10.3,NaN,NaN


In [8]:
# Vérifier chaque condition séparément
condition_rtt_actuel = df_model["rtt_avg_ms"].notna()

condition_rtt_futur = df_model["rtt_future"].notna()

condition_intervalle = (
    df_model["gap_future_s"].notna()
    & (df_model["gap_future_s"] <= 45)
)

# Combiner les trois conditions
masque_valide = (
    condition_rtt_actuel
    & condition_rtt_futur
    & condition_intervalle
)

# Créer le dataset destiné à la modélisation
df_model_clean = df_model.loc[masque_valide].copy()

# Afficher les résultats
print("Dimensions initiales :", df_model.shape)

print("Observations avec RTT actuel disponible :",
      condition_rtt_actuel.sum())

print("Observations avec RTT futur disponible :",
      condition_rtt_futur.sum())

print("Transitions avec intervalle <= 45 secondes :",
      condition_intervalle.sum())

print("\nDimensions après filtrage :", df_model_clean.shape)

print("\nValeurs manquantes dans les deux RTT :")
print(df_model_clean[["rtt_avg_ms", "rtt_future"]].isna().sum())

print("\nStatistiques des intervalles conservés :")
display(df_model_clean["gap_future_s"].describe())

Dimensions initiales : (54617, 21)
Observations avec RTT actuel disponible : 54599
Observations avec RTT futur disponible : 54598
Transitions avec intervalle <= 45 secondes : 54157

Dimensions après filtrage : (54137, 21)

Valeurs manquantes dans les deux RTT :
rtt_avg_ms    0
rtt_future    0
dtype: int64

Statistiques des intervalles conservés :


count    54137.000000
mean        29.994588
std          0.876050
min         10.000000
25%         30.000000
50%         30.000000
75%         30.000000
max         45.000000
Name: gap_future_s, dtype: float64

In [9]:
# Examiner les intervalles inférieurs à 25 secondes
intervalles_courts = df_model_clean[
    df_model_clean["gap_future_s"] < 25
]

print("Nombre d'intervalles inférieurs à 25 secondes :",
      len(intervalles_courts))

print("\nRépartition des intervalles courts :")
display(
    intervalles_courts["gap_future_s"]
    .value_counts()
    .sort_index()
)

print("\nExemples d'observations :")
display(
    intervalles_courts[
        ["timestamp_utc", "rtt_avg_ms",
         "rtt_future", "gap_future_s"]
    ].head(15)
)

Nombre d'intervalles inférieurs à 25 secondes : 143

Répartition des intervalles courts :


gap_future_s
10.0     2
11.0     3
12.0     3
14.0     1
15.0     4
16.0     4
18.0     2
19.0    11
20.0    13
21.0    10
22.0    11
23.0    34
24.0    45
Name: count, dtype: int64


Exemples d'observations :


,timestamp_utc,rtt_avg_ms,rtt_future,gap_future_s
58,2026-09-20 17:05:33+00:00,10.444,24.000,19.0
89,2026-09-20 17:21:33+00:00,12.100,10.600,19.0
119,2026-09-20 17:37:33+00:00,10.500,11.900,19.0
150,2026-09-20 17:53:31+00:00,10.200,10.300,21.0
181,2026-09-20 18:09:32+00:00,10.600,10.667,20.0
212,2026-09-20 18:25:32+00:00,10.700,10.400,20.0
242,2026-09-20 18:41:10+00:00,10.300,10.800,12.0
304,2026-09-20 19:12:37+00:00,12.300,10.900,15.0
335,2026-09-20 19:28:32+00:00,12.500,10.300,20.0
365,2026-09-20 19:44:02+00:00,11.100,10.500,20.0


In [10]:
# Vérifier la disponibilité des RTT
condition_rtt_actuel = df_model["rtt_avg_ms"].notna()
condition_rtt_futur = df_model["rtt_future"].notna()

# Retenir les intervalles compris entre 25 et 45 secondes
condition_intervalle = (
    df_model["gap_future_s"].between(25, 45)
)

# Combiner les conditions
masque_valide = (
    condition_rtt_actuel
    & condition_rtt_futur
    & condition_intervalle
)

# Créer le dataset final de travail
df_model_clean = df_model.loc[masque_valide].copy()

print("Dimensions finales :", df_model_clean.shape)

print("\nStatistiques des intervalles :")
display(df_model_clean["gap_future_s"].describe())

print("\nValeurs manquantes dans les RTT :")
display(
    df_model_clean[["rtt_avg_ms", "rtt_future"]].isna().sum()
)

Dimensions finales : (53994, 21)

Statistiques des intervalles :


count    53994.000000
mean        30.017613
std          0.733106
min         25.000000
25%         30.000000
50%         30.000000
75%         30.000000
max         45.000000
Name: gap_future_s, dtype: float64


Valeurs manquantes dans les RTT :


rtt_avg_ms    0
rtt_future    0
dtype: int64

In [11]:
# Définir les variables explicatives candidates
features = [
    "hour_local",
    "weekday_local",
    "is_weekend",
    "rtt_avg_ms",
    "rtt_min_ms",
    "rtt_max_ms",
    "rtt_median_ms",
    "rtt_stddev_ms",
    "jitter_ms",
    "packet_loss_pct",
    "gateway_rtt_avg_ms",
    "gateway_jitter_ms",
    "gateway_packet_loss_pct",
    "tcp_connect_median_ms",
    "dns_resolve_ms",
    "dns_recursive_ms",
    "http_ms",
    "gap_before_s"
]

# Définir la cible
target = "rtt_future"

# Construire X et y
X = df_model_clean[features].copy()
y = df_model_clean[target].copy()

# Vérifications
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

print("\nValeurs manquantes dans X :")
display(X.isna().sum().sort_values(ascending=False))

print("\nValeurs manquantes dans y :", y.isna().sum())

print("\nAperçu de X :")
display(X.head())

print("\nAperçu de y :")
display(y.head())

Dimensions de X : (53994, 18)
Dimensions de y : (53994,)

Valeurs manquantes dans X :


http_ms                    34
dns_recursive_ms            9
dns_resolve_ms              3
tcp_connect_median_ms       1
weekday_local               0
hour_local                  0
is_weekend                  0
rtt_avg_ms                  0
rtt_min_ms                  0
rtt_max_ms                  0
packet_loss_pct             0
jitter_ms                   0
rtt_stddev_ms               0
rtt_median_ms               0
gateway_packet_loss_pct     0
gateway_jitter_ms           0
gateway_rtt_avg_ms          0
gap_before_s                0
dtype: int64


Valeurs manquantes dans y : 0

Aperçu de X :


,hour_local,weekday_local,is_weekend,rtt_avg_ms,rtt_min_ms,rtt_max_ms,rtt_median_ms,rtt_stddev_ms,jitter_ms,packet_loss_pct,gateway_rtt_avg_ms,gateway_jitter_ms,gateway_packet_loss_pct,tcp_connect_median_ms,dns_resolve_ms,dns_recursive_ms,http_ms,gap_before_s
1,12,6,1,11.222,10.0,12.0,11.0,0.786,0.500,10.0,0.0,0.0,0.0,12.208,132.626,133.356,129.852,90.0
2,12,6,1,10.900,10.0,13.0,11.0,0.943,0.889,0.0,0.4,0.5,0.0,13.766,445.539,97.841,130.988,30.0
3,12,6,1,10.500,10.0,13.0,10.0,0.922,0.889,0.0,0.2,0.5,0.0,28.913,140.461,66.917,1383.776,30.0
4,12,6,1,36.300,10.0,270.0,10.0,77.903,58.000,0.0,0.4,0.5,0.0,13.767,32.678,378.545,331.481,30.0
5,12,6,1,10.800,10.0,13.0,10.5,0.980,1.000,0.0,0.0,0.0,0.0,12.608,516.970,596.208,119.856,30.0



Aperçu de y :


1    10.9
2    10.5
3    36.3
4    10.8
5    12.4
Name: rtt_future, dtype: float64

In [12]:
# Déterminer le point de séparation
n = len(X)
split_index = int(n * 0.8)

# Séparer chronologiquement les données
X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

# Retirer la dernière observation d'entraînement
# pour éviter que sa cible empiète sur le début du test
X_train = X_train.iloc[:-1]
y_train = y_train.iloc[:-1]

# Vérifier les dimensions
print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_test :", X_test.shape)

print("Dimensions de y_train :", y_train.shape)
print("Dimensions de y_test :", y_test.shape)

print("\nPériode d'entraînement :")
print(df_model_clean.iloc[0]["timestamp_utc"])
print(df_model_clean.iloc[split_index - 2]["timestamp_utc"])

print("\nPériode de test :")
print(df_model_clean.iloc[split_index]["timestamp_utc"])
print(df_model_clean.iloc[-1]["timestamp_utc"])

Dimensions de X_train : (43194, 18)
Dimensions de X_test : (10799, 18)
Dimensions de y_train : (43194,)
Dimensions de y_test : (10799,)

Période d'entraînement :
2026-09-20 16:35:52+00:00
2026-10-06 00:15:39+00:00

Période de test :
2026-10-06 00:16:39+00:00
2026-10-09 19:49:13+00:00


In [13]:
from sklearn.impute import SimpleImputer

# Créer l'imputeur utilisant la médiane
imputer = SimpleImputer(strategy="median")

# Calculer les médianes sur les données d'entraînement
X_train_imputed = imputer.fit_transform(X_train)

# Appliquer les mêmes médianes aux données de test
X_test_imputed = imputer.transform(X_test)

# Reconstruire les DataFrames
X_train = pd.DataFrame(
    X_train_imputed,
    columns=X_train.columns,
    index=X_train.index
)

X_test = pd.DataFrame(
    X_test_imputed,
    columns=X_test.columns,
    index=X_test.index
)

# Vérifier le résultat
print("Valeurs manquantes dans X_train :",
      X_train.isna().sum().sum())

print("Valeurs manquantes dans X_test :",
      X_test.isna().sum().sum())

print("\nDimensions finales :")
print("X_train :", X_train.shape)
print("X_test :", X_test.shape)

print("\nMédianes calculées sur Train :")
display(
    pd.Series(
        imputer.statistics_,
        index=X_train.columns,
        name="Médiane"
    )
)

Valeurs manquantes dans X_train : 0
Valeurs manquantes dans X_test : 0

Dimensions finales :
X_train : (43194, 18)
X_test : (10799, 18)

Médianes calculées sur Train :


hour_local                 12.0000
weekday_local               3.0000
is_weekend                  0.0000
rtt_avg_ms                 11.3000
rtt_min_ms                 10.0000
rtt_max_ms                 14.0000
rtt_median_ms              11.0000
rtt_stddev_ms               1.1830
jitter_ms                   1.2220
packet_loss_pct             0.0000
gateway_rtt_avg_ms          0.2000
gateway_jitter_ms           0.2500
gateway_packet_loss_pct     0.0000
tcp_connect_median_ms      13.9160
dns_resolve_ms             36.7445
dns_recursive_ms           61.0250
http_ms                    82.3080
gap_before_s               30.0000
Name: Médiane, dtype: float64

In [14]:
X_train.to_csv("../../data/X_train.csv", index=False)
X_test.to_csv("../../data/X_test.csv", index=False)

y_train.to_csv("../../data/y_train.csv", index=False)
y_test.to_csv("../../data/y_test.csv", index=False)

print("Les quatre fichiers ont été enregistrés.")

Les quatre fichiers ont été enregistrés.


In [15]:
# Repartir des données originales triées
df_lag = df_work.copy()

df_lag = df_lag.sort_values(
    "timestamp_utc"
).reset_index(drop=True)

# Variables dont nous voulons créer les historiques
variables_lag = [
    "rtt_avg_ms",
    "jitter_ms",
    "packet_loss_pct"
]

# Créer les retards de 1, 2 et 3 observations
for variable in variables_lag:
    for lag in [1, 2, 3]:
        nom_colonne = f"{variable}_lag{lag}"
        df_lag[nom_colonne] = df_lag[variable].shift(lag)

# Créer la cible : latence de la prochaine observation
df_lag["rtt_future"] = df_lag["rtt_avg_ms"].shift(-1)

# Intervalle entre l'observation actuelle et la suivante
df_lag["gap_future_s"] = df_lag["gap_before_s"].shift(-1)

# Vérifier que les trois observations historiques
# appartiennent à une séquence temporelle régulière
historique_regulier = (
    df_lag["gap_before_s"].between(25, 45)
    & df_lag["gap_before_s"].shift(1).between(25, 45)
    & df_lag["gap_before_s"].shift(2).between(25, 45)
)

# Vérifier les variables historiques
colonnes_lag = [
    f"{variable}_lag{lag}"
    for variable in variables_lag
    for lag in [1, 2, 3]
]

masque_valide = (
    df_lag["rtt_avg_ms"].notna()
    & df_lag["rtt_future"].notna()
    & df_lag["gap_future_s"].between(25, 45)
    & historique_regulier
    & df_lag[colonnes_lag].notna().all(axis=1)
)

# Conserver uniquement les observations valides
df_lag_clean = df_lag.loc[masque_valide].copy()

print("Dimensions avant filtrage :", df_lag.shape)
print("Dimensions après filtrage :", df_lag_clean.shape)

print("\nValeurs manquantes dans les variables historiques :")
print(df_lag_clean[colonnes_lag].isna().sum())

print("\nExemple des nouvelles variables :")
display(df_lag_clean[
    ["rtt_avg_ms"] + colonnes_lag + ["rtt_future"]
].head(10))

Dimensions avant filtrage : (54617, 30)
Dimensions après filtrage : (52274, 30)

Valeurs manquantes dans les variables historiques :
rtt_avg_ms_lag1         0
rtt_avg_ms_lag2         0
rtt_avg_ms_lag3         0
jitter_ms_lag1          0
jitter_ms_lag2          0
jitter_ms_lag3          0
packet_loss_pct_lag1    0
packet_loss_pct_lag2    0
packet_loss_pct_lag3    0
dtype: int64

Exemple des nouvelles variables :


,rtt_avg_ms,rtt_avg_ms_lag1,rtt_avg_ms_lag2,rtt_avg_ms_lag3,jitter_ms_lag1,jitter_ms_lag2,jitter_ms_lag3,packet_loss_pct_lag1,packet_loss_pct_lag2,packet_loss_pct_lag3,rtt_future
4,36.300,10.5,10.9,11.222,0.889,0.889,0.500,0.0,0.0,10.0,10.800
5,10.800,36.3,10.5,10.900,58.000,0.889,0.889,0.0,0.0,0.0,12.400
6,12.400,10.8,36.3,10.500,1.000,58.000,0.889,0.0,0.0,0.0,11.000
7,11.000,12.4,10.8,36.300,5.111,1.000,58.000,0.0,0.0,0.0,11.000
8,11.000,11.0,12.4,10.800,1.111,5.111,1.000,0.0,0.0,0.0,12.400
9,12.400,11.0,11.0,12.400,0.556,1.111,5.111,0.0,0.0,0.0,10.300
10,10.300,12.4,11.0,11.000,4.667,0.556,1.111,0.0,0.0,0.0,10.400
11,10.400,10.3,12.4,11.000,0.667,4.667,0.556,0.0,0.0,0.0,10.500
12,10.500,10.4,10.3,12.400,0.556,0.667,4.667,0.0,0.0,0.0,10.222
13,10.222,10.5,10.4,10.300,0.667,0.556,0.667,0.0,0.0,0.0,12.000


In [16]:
# Variables mesurées à l'instant actuel
features_courantes = [
    "rtt_avg_ms",
    "rtt_stddev_ms",
    "jitter_ms",
    "packet_loss_pct",
    "gateway_rtt_avg_ms",
    "gateway_jitter_ms"
]

# Variables historiques : lag 1, lag 2 et lag 3
features_lag = [
    f"{variable}_lag{lag}"
    for variable in variables_lag
    for lag in [1, 2, 3]
]

# Combiner les variables actuelles et historiques
features_finales = features_courantes + features_lag

# Construire X et y
X_lag = df_lag_clean[features_finales].copy()
y_lag = df_lag_clean["rtt_future"].copy()

print("Dimensions de X :", X_lag.shape)
print("Dimensions de y :", y_lag.shape)

print("\nValeurs manquantes dans X :")
print(X_lag.isna().sum())

print("\nValeurs manquantes dans y :", y_lag.isna().sum())

Dimensions de X : (52274, 15)
Dimensions de y : (52274,)

Valeurs manquantes dans X :
rtt_avg_ms              0
rtt_stddev_ms           0
jitter_ms               0
packet_loss_pct         0
gateway_rtt_avg_ms      0
gateway_jitter_ms       0
rtt_avg_ms_lag1         0
rtt_avg_ms_lag2         0
rtt_avg_ms_lag3         0
jitter_ms_lag1          0
jitter_ms_lag2          0
jitter_ms_lag3          0
packet_loss_pct_lag1    0
packet_loss_pct_lag2    0
packet_loss_pct_lag3    0
dtype: int64

Valeurs manquantes dans y : 0


In [17]:
# Déterminer la position de séparation
split_index = int(len(X_lag) * 0.8)

# Séparer les données chronologiquement
X_train_lag = X_lag.iloc[:split_index].copy()
X_test_lag = X_lag.iloc[split_index:].copy()

y_train_lag = y_lag.iloc[:split_index].copy()
y_test_lag = y_lag.iloc[split_index:].copy()

# Retirer la dernière observation d'entraînement
# pour éviter le chevauchement avec la première cible du test
X_train_lag = X_train_lag.iloc[:-1]
y_train_lag = y_train_lag.iloc[:-1]

print("X_train :", X_train_lag.shape)
print("X_test  :", X_test_lag.shape)
print("y_train :", y_train_lag.shape)
print("y_test  :", y_test_lag.shape)

X_train : (41818, 15)
X_test  : (10455, 15)
y_train : (41818,)
y_test  : (10455,)


In [18]:
X_train_lag.to_csv("../../data/X_train_lag.csv", index=False)
X_test_lag.to_csv("../../data/X_test_lag.csv", index=False)

y_train_lag.to_csv("../../data/y_train_lag.csv", index=False)
y_test_lag.to_csv("../../data/y_test_lag.csv", index=False)

print("Les quatre fichiers ont été sauvegardés.")

Les quatre fichiers ont été sauvegardés.


In [19]:
# Repartir des données originales
df_lag5 = df_work.copy()

df_lag5 = df_lag5.sort_values(
    "timestamp_utc"
).reset_index(drop=True)

# Variables historiques
variables_lag5 = [
    "rtt_avg_ms",
    "jitter_ms",
    "packet_loss_pct"
]

# Créer les historiques de 1 à 5 observations
for variable in variables_lag5:
    for lag in range(1, 6):
        nom_colonne = f"{variable}_lag{lag}"
        df_lag5[nom_colonne] = df_lag5[variable].shift(lag)

# Créer la cible : latence de la prochaine observation
df_lag5["rtt_future"] = df_lag5["rtt_avg_ms"].shift(-1)

# Intervalle jusqu'à la prochaine observation
df_lag5["gap_future_s"] = df_lag5["gap_before_s"].shift(-1)

# Vérifier la régularité des 5 observations historiques
historique_regulier_5 = df_lag5["gap_before_s"].between(25, 45)

for decalage in range(1, 5):
    historique_regulier_5 = (
        historique_regulier_5
        & df_lag5["gap_before_s"].shift(decalage).between(25, 45)
    )

# Construire la liste des colonnes historiques
colonnes_lag5 = [
    f"{variable}_lag{lag}"
    for variable in variables_lag5
    for lag in range(1, 6)
]

# Conditions pour conserver une observation
masque_valide_5 = (
    df_lag5["rtt_avg_ms"].notna()
    & df_lag5["rtt_future"].notna()
    & df_lag5["gap_future_s"].between(25, 45)
    & historique_regulier_5
    & df_lag5[colonnes_lag5].notna().all(axis=1)
)

# Conserver les observations valides
df_lag5_clean = df_lag5.loc[masque_valide_5].copy()

print("Dimensions avant filtrage :", df_lag5.shape)
print("Dimensions après filtrage :", df_lag5_clean.shape)

print("\nValeurs manquantes dans les historiques :")
print(df_lag5_clean[colonnes_lag5].isna().sum())

print("\nExemple des nouvelles variables :")
display(
    df_lag5_clean[
        ["rtt_avg_ms"]
        + colonnes_lag5
        + ["rtt_future"]
    ].head(5)
)

Dimensions avant filtrage : (54617, 36)
Dimensions après filtrage : (51137, 36)

Valeurs manquantes dans les historiques :
rtt_avg_ms_lag1         0
rtt_avg_ms_lag2         0
rtt_avg_ms_lag3         0
rtt_avg_ms_lag4         0
rtt_avg_ms_lag5         0
jitter_ms_lag1          0
jitter_ms_lag2          0
jitter_ms_lag3          0
jitter_ms_lag4          0
jitter_ms_lag5          0
packet_loss_pct_lag1    0
packet_loss_pct_lag2    0
packet_loss_pct_lag3    0
packet_loss_pct_lag4    0
packet_loss_pct_lag5    0
dtype: int64

Exemple des nouvelles variables :


,rtt_avg_ms,rtt_avg_ms_lag1,rtt_avg_ms_lag2,rtt_avg_ms_lag3,rtt_avg_ms_lag4,rtt_avg_ms_lag5,jitter_ms_lag1,jitter_ms_lag2,jitter_ms_lag3,jitter_ms_lag4,jitter_ms_lag5,packet_loss_pct_lag1,packet_loss_pct_lag2,packet_loss_pct_lag3,packet_loss_pct_lag4,packet_loss_pct_lag5,rtt_future
6,12.4,10.8,36.3,10.5,10.9,11.222,1.000,58.000,0.889,0.889,0.500,0.0,0.0,0.0,0.0,10.0,11.0
7,11.0,12.4,10.8,36.3,10.5,10.900,5.111,1.000,58.000,0.889,0.889,0.0,0.0,0.0,0.0,0.0,11.0
8,11.0,11.0,12.4,10.8,36.3,10.500,1.111,5.111,1.000,58.000,0.889,0.0,0.0,0.0,0.0,0.0,12.4
9,12.4,11.0,11.0,12.4,10.8,36.300,0.556,1.111,5.111,1.000,58.000,0.0,0.0,0.0,0.0,0.0,10.3
10,10.3,12.4,11.0,11.0,12.4,10.800,4.667,0.556,1.111,5.111,1.000,0.0,0.0,0.0,0.0,0.0,10.4


In [20]:
features_courantes = [
    "rtt_avg_ms",
    "rtt_stddev_ms",
    "jitter_ms",
    "packet_loss_pct",
    "gateway_rtt_avg_ms",
    "gateway_jitter_ms"
]

features_lag5 = [
    f"{variable}_lag{lag}"
    for variable in variables_lag5
    for lag in range(1, 6)
]

features_finales_5 = features_courantes + features_lag5

X_lag5 = df_lag5_clean[features_finales_5].copy()
y_lag5 = df_lag5_clean["rtt_future"].copy()

print("Dimensions de X :", X_lag5.shape)
print("Dimensions de y :", y_lag5.shape)

print("\nValeurs manquantes dans X :")
print(X_lag5.isna().sum())

print("\nValeurs manquantes dans y :", y_lag5.isna().sum())

Dimensions de X : (51137, 21)
Dimensions de y : (51137,)

Valeurs manquantes dans X :
rtt_avg_ms              0
rtt_stddev_ms           0
jitter_ms               0
packet_loss_pct         0
gateway_rtt_avg_ms      0
gateway_jitter_ms       0
rtt_avg_ms_lag1         0
rtt_avg_ms_lag2         0
rtt_avg_ms_lag3         0
rtt_avg_ms_lag4         0
rtt_avg_ms_lag5         0
jitter_ms_lag1          0
jitter_ms_lag2          0
jitter_ms_lag3          0
jitter_ms_lag4          0
jitter_ms_lag5          0
packet_loss_pct_lag1    0
packet_loss_pct_lag2    0
packet_loss_pct_lag3    0
packet_loss_pct_lag4    0
packet_loss_pct_lag5    0
dtype: int64

Valeurs manquantes dans y : 0


In [21]:
split_index_5 = int(len(X_lag5) * 0.8)

X_train_lag5 = X_lag5.iloc[:split_index_5].copy()
X_test_lag5 = X_lag5.iloc[split_index_5:].copy()

y_train_lag5 = y_lag5.iloc[:split_index_5].copy()
y_test_lag5 = y_lag5.iloc[split_index_5:].copy()

# Éviter le chevauchement entre entraînement et test
X_train_lag5 = X_train_lag5.iloc[:-1]
y_train_lag5 = y_train_lag5.iloc[:-1]

print("X_train :", X_train_lag5.shape)
print("X_test :", X_test_lag5.shape)
print("y_train :", y_train_lag5.shape)
print("y_test :", y_test_lag5.shape)

X_train : (40908, 21)
X_test : (10228, 21)
y_train : (40908,)
y_test : (10228,)


In [22]:
X_train_lag5.to_csv("../../data/X_train_lag5.csv", index=False)
X_test_lag5.to_csv("../../data/X_test_lag5.csv", index=False)

y_train_lag5.to_csv("../../data/y_train_lag5.csv", index=False)
y_test_lag5.to_csv("../../data/y_test_lag5.csv", index=False)

print("Les fichiers avec 5 historiques sont sauvegardés.")

Les fichiers avec 5 historiques sont sauvegardés.


In [23]:
# Identifier les observations présentes dans les deux préparations
indices_communs = (
    df_lag_clean.index
    .intersection(df_lag5_clean.index)
    .sort_values()
)

# Variables actuelles
features_courantes = [
    "rtt_avg_ms",
    "rtt_stddev_ms",
    "jitter_ms",
    "packet_loss_pct",
    "gateway_rtt_avg_ms",
    "gateway_jitter_ms"
]

# Variables historiques pour 3 observations
features_lag3 = [
    f"{variable}_lag{lag}"
    for variable in variables_lag
    for lag in range(1, 4)
]

# Variables historiques pour 5 observations
features_lag5 = [
    f"{variable}_lag{lag}"
    for variable in variables_lag5
    for lag in range(1, 6)
]

# Construire les deux jeux de variables sur les mêmes lignes
X3_commun = df_lag_clean.loc[
    indices_communs,
    features_courantes + features_lag3
].copy()

X5_commun = df_lag5_clean.loc[
    indices_communs,
    features_courantes + features_lag5
].copy()

# Cible commune
y_commun = df_lag5_clean.loc[
    indices_communs, "rtt_future"
].copy()

# Vérifications
assert X3_commun.index.equals(X5_commun.index)
assert X3_commun.index.equals(y_commun.index)

print("Nombre d'observations communes :", len(indices_communs))
print("X3 :", X3_commun.shape)
print("X5 :", X5_commun.shape)
print("y :", y_commun.shape)

Nombre d'observations communes : 51137
X3 : (51137, 15)
X5 : (51137, 21)
y : (51137,)


In [24]:
split_commun = int(len(X3_commun) * 0.8)

X3_train = X3_commun.iloc[:split_commun].copy()
X3_test = X3_commun.iloc[split_commun:].copy()

X5_train = X5_commun.iloc[:split_commun].copy()
X5_test = X5_commun.iloc[split_commun:].copy()

y_train_commun = y_commun.iloc[:split_commun].copy()
y_test_commun = y_commun.iloc[split_commun:].copy()

# Éviter le chevauchement temporel entre train et test
X3_train = X3_train.iloc[:-1]
X5_train = X5_train.iloc[:-1]
y_train_commun = y_train_commun.iloc[:-1]

print("Entraînement RF 3 :", X3_train.shape)
print("Entraînement RF 5 :", X5_train.shape)
print("Test RF 3 :", X3_test.shape)
print("Test RF 5 :", X5_test.shape)
print("Cible entraînement :", y_train_commun.shape)
print("Cible test :", y_test_commun.shape)

Entraînement RF 3 : (40908, 15)
Entraînement RF 5 : (40908, 21)
Test RF 3 : (10228, 15)
Test RF 5 : (10228, 21)
Cible entraînement : (40908,)
Cible test : (10228,)


In [25]:
X3_train.to_csv("../../data/X3_train_commun.csv", index=False)
X3_test.to_csv("../../data/X3_test_commun.csv", index=False)

X5_train.to_csv("../../data/X5_train_commun.csv", index=False)
X5_test.to_csv("../../data/X5_test_commun.csv", index=False)

y_train_commun.to_csv("../../data/y_train_commun.csv", index=False)
y_test_commun.to_csv("../../data/y_test_commun.csv", index=False)

print("Jeux de comparaison communs sauvegardés.")

Jeux de comparaison communs sauvegardés.


In [27]:

# Création d'une copie des données originales
df_lag9 = df_work.copy()

# Création de la cible : RTT de la prochaine observation
df_lag9["rtt_future"] = df_lag9["rtt_avg_ms"].shift(-1)

# Intervalle avant la prochaine observation
df_lag9["gap_future_s"] = df_lag9["gap_before_s"].shift(-1)

# Création des 9 historiques
variables_historiques = [
    "rtt_avg_ms",
    "jitter_ms",
    "packet_loss_pct"
]

for variable in variables_historiques:
    for lag in range(1, 10):
        df_lag9[f"{variable}_lag{lag}"] = df_lag9[variable].shift(lag)

print("Dimensions :", df_lag9.shape)
print("\nDernières colonnes :")
print(df_lag9.columns[-30:].tolist())


Dimensions : (54617, 48)

Dernières colonnes :
['gap_before_s', 'rtt_future', 'gap_future_s', 'rtt_avg_ms_lag1', 'rtt_avg_ms_lag2', 'rtt_avg_ms_lag3', 'rtt_avg_ms_lag4', 'rtt_avg_ms_lag5', 'rtt_avg_ms_lag6', 'rtt_avg_ms_lag7', 'rtt_avg_ms_lag8', 'rtt_avg_ms_lag9', 'jitter_ms_lag1', 'jitter_ms_lag2', 'jitter_ms_lag3', 'jitter_ms_lag4', 'jitter_ms_lag5', 'jitter_ms_lag6', 'jitter_ms_lag7', 'jitter_ms_lag8', 'jitter_ms_lag9', 'packet_loss_pct_lag1', 'packet_loss_pct_lag2', 'packet_loss_pct_lag3', 'packet_loss_pct_lag4', 'packet_loss_pct_lag5', 'packet_loss_pct_lag6', 'packet_loss_pct_lag7', 'packet_loss_pct_lag8', 'packet_loss_pct_lag9']


In [28]:

# Vérifier que les 9 observations historiques
# sont séparées par des intervalles réguliers

condition_intervalles = df_lag9["gap_before_s"].between(25, 45)

for lag in range(1, 9):
    condition_intervalles &= (
        df_lag9["gap_before_s"].shift(lag).between(25, 45)
    )

# Vérifier les valeurs RTT actuelles et futures
condition_rtt_actuel = df_lag9["rtt_avg_ms"].notna()
condition_rtt_futur = df_lag9["rtt_future"].notna()

# Vérifier que la prochaine observation arrive
# dans un intervalle compris entre 25 et 45 secondes
condition_intervalle_futur = df_lag9["gap_future_s"].between(25, 45)

# Vérifier que les 27 variables historiques sont disponibles
colonnes_lags = [
    f"{variable}_lag{lag}"
    for variable in ["rtt_avg_ms", "jitter_ms", "packet_loss_pct"]
    for lag in range(1, 10)
]

condition_lags = df_lag9[colonnes_lags].notna().all(axis=1)

# Combiner toutes les conditions
masque_valide_9 = (
    condition_intervalles
    & condition_rtt_actuel
    & condition_rtt_futur
    & condition_intervalle_futur
    & condition_lags
)

df_lag9_clean = df_lag9.loc[masque_valide_9].copy()

print("Dimensions avant filtrage :", df_lag9.shape)
print("Dimensions après filtrage :", df_lag9_clean.shape)

print("\nValeurs manquantes dans les historiques :")
print(df_lag9_clean[colonnes_lags].isna().sum().sum())

print("\nPourcentage de données conservées :",
      round(len(df_lag9_clean) / len(df_lag9) * 100, 2), "%")


Dimensions avant filtrage : (54617, 48)
Dimensions après filtrage : (48900, 48)

Valeurs manquantes dans les historiques :
0

Pourcentage de données conservées : 89.53 %


In [29]:

# Les 6 variables actuelles
features_actuelles = [
    "rtt_avg_ms",
    "rtt_stddev_ms",
    "jitter_ms",
    "packet_loss_pct",
    "gateway_rtt_avg_ms",
    "gateway_jitter_ms"
]

# Les 27 variables historiques
features_historiques_9 = [
    f"{variable}_lag{lag}"
    for variable in [
        "rtt_avg_ms",
        "jitter_ms",
        "packet_loss_pct"
    ]
    for lag in range(1, 10)
]

# Les 33 variables finales
features_lag9 = features_actuelles + features_historiques_9

# Construction de X et y
X_lag9 = df_lag9_clean[features_lag9].copy()
y_lag9 = df_lag9_clean["rtt_future"].copy()

print("Dimensions de X :", X_lag9.shape)
print("Dimensions de y :", y_lag9.shape)

print("\nNombre de variables :", len(features_lag9))
print("\nValeurs manquantes dans X :", X_lag9.isna().sum().sum())
print("Valeurs manquantes dans y :", y_lag9.isna().sum())


Dimensions de X : (48900, 33)
Dimensions de y : (48900,)

Nombre de variables : 33

Valeurs manquantes dans X : 0
Valeurs manquantes dans y : 0


In [30]:

# Création du dataset avec 8 historiques
df_lag8 = df_work.copy()

# Cible : RTT de la prochaine observation
df_lag8["rtt_future"] = df_lag8["rtt_avg_ms"].shift(-1)
df_lag8["gap_future_s"] = df_lag8["gap_before_s"].shift(-1)

# Création des 8 historiques
for variable in ["rtt_avg_ms", "jitter_ms", "packet_loss_pct"]:
    for lag in range(1, 9):
        df_lag8[f"{variable}_lag{lag}"] = df_lag8[variable].shift(lag)

# Vérification des intervalles
condition_intervalles_8 = df_lag8["gap_before_s"].between(25, 45)

for lag in range(1, 8):
    condition_intervalles_8 &= (
        df_lag8["gap_before_s"].shift(lag).between(25, 45)
    )

# Conditions de validité
condition_rtt_actuel_8 = df_lag8["rtt_avg_ms"].notna()
condition_rtt_futur_8 = df_lag8["rtt_future"].notna()
condition_intervalle_futur_8 = df_lag8["gap_future_s"].between(25, 45)

colonnes_lags_8 = [
    f"{variable}_lag{lag}"
    for variable in ["rtt_avg_ms", "jitter_ms", "packet_loss_pct"]
    for lag in range(1, 9)
]

condition_lags_8 = df_lag8[colonnes_lags_8].notna().all(axis=1)

# Filtrage final
masque_valide_8 = (
    condition_intervalles_8
    & condition_rtt_actuel_8
    & condition_rtt_futur_8
    & condition_intervalle_futur_8
    & condition_lags_8
)

df_lag8_clean = df_lag8.loc[masque_valide_8].copy()

print("Dimensions avant filtrage :", df_lag8.shape)
print("Dimensions après filtrage :", df_lag8_clean.shape)
print("Valeurs manquantes dans les historiques :",
      df_lag8_clean[colonnes_lags_8].isna().sum().sum())


Dimensions avant filtrage : (54617, 45)
Dimensions après filtrage : (49457, 45)
Valeurs manquantes dans les historiques : 0


In [31]:

features_historiques_8 = [
    f"{variable}_lag{lag}"
    for variable in ["rtt_avg_ms", "jitter_ms", "packet_loss_pct"]
    for lag in range(1, 9)
]

features_lag8 = features_actuelles + features_historiques_8

X_lag8 = df_lag8_clean[features_lag8].copy()
y_lag8 = df_lag8_clean["rtt_future"].copy()

print("Dimensions de X :", X_lag8.shape)
print("Dimensions de y :", y_lag8.shape)
print("Nombre de variables :", len(features_lag8))
print("Valeurs manquantes dans X :", X_lag8.isna().sum().sum())
print("Valeurs manquantes dans y :", y_lag8.isna().sum())


Dimensions de X : (49457, 30)
Dimensions de y : (49457,)
Nombre de variables : 30
Valeurs manquantes dans X : 0
Valeurs manquantes dans y : 0


In [32]:

# Indices communs aux quatre datasets
indices_communs_4 = (
    df_lag_clean.index
    .intersection(df_lag5_clean.index)
    .intersection(df_lag8_clean.index)
    .intersection(df_lag9_clean.index)
    .sort_values()
)

print("Nombre d'observations communes :", len(indices_communs_4))

# Construction des datasets communs
X3_commun = df_lag_clean.loc[indices_communs_4, features_lag3].copy()
X5_commun = df_lag5_clean.loc[indices_communs_4, features_lag5].copy()
X8_commun = df_lag8_clean.loc[indices_communs_4, features_lag8].copy()
X9_commun = df_lag9_clean.loc[indices_communs_4, features_lag9].copy()

# Une seule cible commune
y_commun = df_lag9_clean.loc[indices_communs_4, "rtt_future"].copy()

print("\nDimensions des datasets :")
print("X3 :", X3_commun.shape)
print("X5 :", X5_commun.shape)
print("X8 :", X8_commun.shape)
print("X9 :", X9_commun.shape)
print("y  :", y_commun.shape)

# Vérification de l'alignement des cibles
print("\nCibles identiques :",
      y_commun.equals(df_lag8_clean.loc[indices_communs_4, "rtt_future"])
      and y_commun.equals(df_lag_clean.loc[indices_communs_4, "rtt_future"]))


Nombre d'observations communes : 48900

Dimensions des datasets :
X3 : (48900, 9)
X5 : (48900, 15)
X8 : (48900, 30)
X9 : (48900, 33)
y  : (48900,)

Cibles identiques : True


In [33]:

print("RF3 :", len(features_lag3), features_lag3)
print("\nRF5 :", len(features_lag5), features_lag5)
print("\nRF8 :", len(features_lag8), features_lag8)
print("\nRF9 :", len(features_lag9), features_lag9)


RF3 : 9 ['rtt_avg_ms_lag1', 'rtt_avg_ms_lag2', 'rtt_avg_ms_lag3', 'jitter_ms_lag1', 'jitter_ms_lag2', 'jitter_ms_lag3', 'packet_loss_pct_lag1', 'packet_loss_pct_lag2', 'packet_loss_pct_lag3']

RF5 : 15 ['rtt_avg_ms_lag1', 'rtt_avg_ms_lag2', 'rtt_avg_ms_lag3', 'rtt_avg_ms_lag4', 'rtt_avg_ms_lag5', 'jitter_ms_lag1', 'jitter_ms_lag2', 'jitter_ms_lag3', 'jitter_ms_lag4', 'jitter_ms_lag5', 'packet_loss_pct_lag1', 'packet_loss_pct_lag2', 'packet_loss_pct_lag3', 'packet_loss_pct_lag4', 'packet_loss_pct_lag5']

RF8 : 30 ['rtt_avg_ms', 'rtt_stddev_ms', 'jitter_ms', 'packet_loss_pct', 'gateway_rtt_avg_ms', 'gateway_jitter_ms', 'rtt_avg_ms_lag1', 'rtt_avg_ms_lag2', 'rtt_avg_ms_lag3', 'rtt_avg_ms_lag4', 'rtt_avg_ms_lag5', 'rtt_avg_ms_lag6', 'rtt_avg_ms_lag7', 'rtt_avg_ms_lag8', 'jitter_ms_lag1', 'jitter_ms_lag2', 'jitter_ms_lag3', 'jitter_ms_lag4', 'jitter_ms_lag5', 'jitter_ms_lag6', 'jitter_ms_lag7', 'jitter_ms_lag8', 'packet_loss_pct_lag1', 'packet_loss_pct_lag2', 'packet_loss_pct_lag3', 'packet

In [34]:

# Ajouter les 6 variables actuelles aux RF3 et RF5

features_lag3_completes = features_actuelles + features_lag3
features_lag5_completes = features_actuelles + features_lag5

# Reconstruire les quatre datasets communs
X3_commun = df_lag_clean.loc[
    indices_communs_4, features_lag3_completes
].copy()

X5_commun = df_lag5_clean.loc[
    indices_communs_4, features_lag5_completes
].copy()

X8_commun = df_lag8_clean.loc[
    indices_communs_4, features_lag8
].copy()

X9_commun = df_lag9_clean.loc[
    indices_communs_4, features_lag9
].copy()

# Cible commune
y_commun = df_lag9_clean.loc[
    indices_communs_4, "rtt_future"
].copy()

# Vérification
print("RF3 :", X3_commun.shape)
print("RF5 :", X5_commun.shape)
print("RF8 :", X8_commun.shape)
print("RF9 :", X9_commun.shape)
print("y   :", y_commun.shape)

print("\nValeurs manquantes :")
print(
    X3_commun.isna().sum().sum(),
    X5_commun.isna().sum().sum(),
    X8_commun.isna().sum().sum(),
    X9_commun.isna().sum().sum(),
    y_commun.isna().sum()
)


RF3 : (48900, 15)
RF5 : (48900, 21)
RF8 : (48900, 30)
RF9 : (48900, 33)
y   : (48900,)

Valeurs manquantes :
0 0 0 0 0


In [36]:

# Déterminer l'indice de séparation
split_index = int(len(X9_commun) * 0.8)

# Fonction pour découper les données
def decouper_dataset(X, y, split_index):
    X_train = X.iloc[:split_index].copy()
    X_test = X.iloc[split_index:].copy()

    y_train = y.iloc[:split_index].copy()
    y_test = y.iloc[split_index:].copy()

    # Retirer la dernière observation du train
    X_train = X_train.iloc[:-1]
    y_train = y_train.iloc[:-1]

    return X_train, X_test, y_train, y_test

# Découpage des quatre datasets
X3_train, X3_test, y3_train, y3_test = decouper_dataset(
    X3_commun, y_commun, split_index
)

X5_train, X5_test, y5_train, y5_test = decouper_dataset(
    X5_commun, y_commun, split_index
)

X8_train, X8_test, y8_train, y8_test = decouper_dataset(
    X8_commun, y_commun, split_index
)

X9_train, X9_test, y9_train, y9_test = decouper_dataset(
    X9_commun, y_commun, split_index
)

# Vérification des dimensions
for nom, Xtr, Xte, ytr, yte in [
    ("RF3", X3_train, X3_test, y3_train, y3_test),
    ("RF5", X5_train, X5_test, y5_train, y5_test),
    ("RF8", X8_train, X8_test, y8_train, y8_test),
    ("RF9", X9_train, X9_test, y9_train, y9_test)
]:
    print(f"\n{nom}")
    print("X_train :", Xtr.shape)
    print("X_test  :", Xte.shape)
    print("y_train :", ytr.shape)
    print("y_test  :", yte.shape)

# Vérifier que toutes les cibles sont identiques
print("\nCibles train identiques :",
      y3_train.equals(y5_train)
      and y3_train.equals(y8_train)
      and y3_train.equals(y9_train))

print("Cibles test identiques :",
      y3_test.equals(y5_test)
      and y3_test.equals(y8_test)
      and y3_test.equals(y9_test))



RF3
X_train : (39119, 15)
X_test  : (9780, 15)
y_train : (39119,)
y_test  : (9780,)

RF5
X_train : (39119, 21)
X_test  : (9780, 21)
y_train : (39119,)
y_test  : (9780,)

RF8
X_train : (39119, 30)
X_test  : (9780, 30)
y_train : (39119,)
y_test  : (9780,)

RF9
X_train : (39119, 33)
X_test  : (9780, 33)
y_train : (39119,)
y_test  : (9780,)

Cibles train identiques : True
Cibles test identiques : True


In [37]:

# Sauvegarde des datasets communs pour les quatre modèles

X3_train.to_csv("../../data/X3_train_commun.csv", index=False)
X3_test.to_csv("../../data/X3_test_commun.csv", index=False)

X5_train.to_csv("../../data/X5_train_commun.csv", index=False)
X5_test.to_csv("../../data/X5_test_commun.csv", index=False)

X8_train.to_csv("../../data/X8_train_commun.csv", index=False)
X8_test.to_csv("../../data/X8_test_commun.csv", index=False)

X9_train.to_csv("../../data/X9_train_commun.csv", index=False)
X9_test.to_csv("../../data/X9_test_commun.csv", index=False)

y3_train.to_csv("../../data/y_train_4modeles.csv", index=False)
y3_test.to_csv("../../data/y_test_4modeles.csv", index=False)

print("Tous les datasets ont été sauvegardés !")


Tous les datasets ont été sauvegardés !


In [38]:

# Création du dataset RF12
df_lag12 = df_work.copy()

# Cible : RTT de la prochaine observation
df_lag12["rtt_future"] = df_lag12["rtt_avg_ms"].shift(-1)
df_lag12["gap_future_s"] = df_lag12["gap_before_s"].shift(-1)

# Création des 12 historiques
for variable in ["rtt_avg_ms", "jitter_ms", "packet_loss_pct"]:
    for lag in range(1, 13):
        df_lag12[f"{variable}_lag{lag}"] = (
            df_lag12[variable].shift(lag)
        )

print("Dimensions :", df_lag12.shape)
print("Nombre de variables historiques :", 36)


Dimensions : (54617, 57)
Nombre de variables historiques : 36


In [39]:

# Vérification des 12 intervalles historiques
condition_intervalles_12 = df_lag12["gap_before_s"].between(25, 45)

for lag in range(1, 12):
    condition_intervalles_12 &= (
        df_lag12["gap_before_s"].shift(lag).between(25, 45)
    )

# Vérification des RTT actuels et futurs
condition_rtt_actuel_12 = df_lag12["rtt_avg_ms"].notna()
condition_rtt_futur_12 = df_lag12["rtt_future"].notna()

# Vérification du prochain intervalle
condition_intervalle_futur_12 = (
    df_lag12["gap_future_s"].between(25, 45)
)

# Vérification des 36 variables historiques
colonnes_lags_12 = [
    f"{variable}_lag{lag}"
    for variable in ["rtt_avg_ms", "jitter_ms", "packet_loss_pct"]
    for lag in range(1, 13)
]

condition_lags_12 = df_lag12[colonnes_lags_12].notna().all(axis=1)

# Filtrage final
masque_valide_12 = (
    condition_intervalles_12
    & condition_rtt_actuel_12
    & condition_rtt_futur_12
    & condition_intervalle_futur_12
    & condition_lags_12
)

df_lag12_clean = df_lag12.loc[masque_valide_12].copy()

print("Dimensions avant filtrage :", df_lag12.shape)
print("Dimensions après filtrage :", df_lag12_clean.shape)

print(
    "Valeurs manquantes dans les historiques :",
    df_lag12_clean[colonnes_lags_12].isna().sum().sum()
)

print(
    "Pourcentage conservé :",
    round(len(df_lag12_clean) / len(df_lag12) * 100, 2),
    "%"
)


Dimensions avant filtrage : (54617, 57)
Dimensions après filtrage : (47244, 57)
Valeurs manquantes dans les historiques : 0
Pourcentage conservé : 86.5 %


In [40]:

# Les 36 variables historiques
features_historiques_12 = [
    f"{variable}_lag{lag}"
    for variable in ["rtt_avg_ms", "jitter_ms", "packet_loss_pct"]
    for lag in range(1, 13)
]

# Les 42 variables finales
features_lag12 = features_actuelles + features_historiques_12

# Construction de X et y
X_lag12 = df_lag12_clean[features_lag12].copy()
y_lag12 = df_lag12_clean["rtt_future"].copy()

# Vérification
print("Dimensions de X :", X_lag12.shape)
print("Dimensions de y :", y_lag12.shape)

print("Nombre de variables :", len(features_lag12))
print("Valeurs manquantes dans X :", X_lag12.isna().sum().sum())
print("Valeurs manquantes dans y :", y_lag12.isna().sum())


Dimensions de X : (47244, 42)
Dimensions de y : (47244,)
Nombre de variables : 42
Valeurs manquantes dans X : 0
Valeurs manquantes dans y : 0


In [41]:

# Intersection des observations valides des cinq modèles
indices_communs_5 = (
    df_lag_clean.index
    .intersection(df_lag5_clean.index)
    .intersection(df_lag8_clean.index)
    .intersection(df_lag9_clean.index)
    .intersection(df_lag12_clean.index)
    .sort_values()
)

print("Observations communes :", len(indices_communs_5))

# Construction des datasets
X3_commun = df_lag_clean.loc[
    indices_communs_5, features_lag3_completes
].copy()

X5_commun = df_lag5_clean.loc[
    indices_communs_5, features_lag5_completes
].copy()

X8_commun = df_lag8_clean.loc[
    indices_communs_5, features_lag8
].copy()

X9_commun = df_lag9_clean.loc[
    indices_communs_5, features_lag9
].copy()

X12_commun = df_lag12_clean.loc[
    indices_communs_5, features_lag12
].copy()

# Cible commune
y_commun = df_lag12_clean.loc[
    indices_communs_5, "rtt_future"
].copy()

# Vérification des dimensions
print("\nDimensions :")
print("RF3  :", X3_commun.shape)
print("RF5  :", X5_commun.shape)
print("RF8  :", X8_commun.shape)
print("RF9  :", X9_commun.shape)
print("RF12 :", X12_commun.shape)
print("y    :", y_commun.shape)

# Vérification des cibles
print("\nCibles identiques :",
      y_commun.equals(
          df_lag_clean.loc[indices_communs_5, "rtt_future"]
      )
      and y_commun.equals(
          df_lag9_clean.loc[indices_communs_5, "rtt_future"]
      ))


Observations communes : 47244

Dimensions :
RF3  : (47244, 15)
RF5  : (47244, 21)
RF8  : (47244, 30)
RF9  : (47244, 33)
RF12 : (47244, 42)
y    : (47244,)

Cibles identiques : True


In [42]:

# Indice de séparation
split_index = int(len(X12_commun) * 0.8)

# Fonction de découpage
def decouper_dataset(X, y, split_index):
    X_train = X.iloc[:split_index].copy()
    X_test = X.iloc[split_index:].copy()

    y_train = y.iloc[:split_index].copy()
    y_test = y.iloc[split_index:].copy()

    # Éviter le chevauchement entre train et test
    X_train = X_train.iloc[:-1]
    y_train = y_train.iloc[:-1]

    return X_train, X_test, y_train, y_test

# Découpage des cinq datasets
X3_train, X3_test, y3_train, y3_test = decouper_dataset(
    X3_commun, y_commun, split_index
)

X5_train, X5_test, y5_train, y5_test = decouper_dataset(
    X5_commun, y_commun, split_index
)

X8_train, X8_test, y8_train, y8_test = decouper_dataset(
    X8_commun, y_commun, split_index
)

X9_train, X9_test, y9_train, y9_test = decouper_dataset(
    X9_commun, y_commun, split_index
)

X12_train, X12_test, y12_train, y12_test = decouper_dataset(
    X12_commun, y_commun, split_index
)

# Vérification
for nom, Xtr, Xte, ytr, yte in [
    ("RF3", X3_train, X3_test, y3_train, y3_test),
    ("RF5", X5_train, X5_test, y5_train, y5_test),
    ("RF8", X8_train, X8_test, y8_train, y8_test),
    ("RF9", X9_train, X9_test, y9_train, y9_test),
    ("RF12", X12_train, X12_test, y12_train, y12_test)
]:
    print(f"\n{nom}")
    print("X_train :", Xtr.shape)
    print("X_test  :", Xte.shape)
    print("y_train :", ytr.shape)
    print("y_test  :", yte.shape)

print("\nCibles train identiques :",
      y3_train.equals(y5_train)
      and y3_train.equals(y8_train)
      and y3_train.equals(y9_train)
      and y3_train.equals(y12_train))

print("Cibles test identiques :",
      y3_test.equals(y5_test)
      and y3_test.equals(y8_test)
      and y3_test.equals(y9_test)
      and y3_test.equals(y12_test))



RF3
X_train : (37794, 15)
X_test  : (9449, 15)
y_train : (37794,)
y_test  : (9449,)

RF5
X_train : (37794, 21)
X_test  : (9449, 21)
y_train : (37794,)
y_test  : (9449,)

RF8
X_train : (37794, 30)
X_test  : (9449, 30)
y_train : (37794,)
y_test  : (9449,)

RF9
X_train : (37794, 33)
X_test  : (9449, 33)
y_train : (37794,)
y_test  : (9449,)

RF12
X_train : (37794, 42)
X_test  : (9449, 42)
y_train : (37794,)
y_test  : (9449,)

Cibles train identiques : True
Cibles test identiques : True


In [43]:

# Sauvegarde des datasets des cinq modèles
for nom, Xtr, Xte in [
    ("3", X3_train, X3_test),
    ("5", X5_train, X5_test),
    ("8", X8_train, X8_test),
    ("9", X9_train, X9_test),
    ("12", X12_train, X12_test)
]:
    Xtr.to_csv(f"../../data/X{nom}_train_commun_5.csv", index=False)
    Xte.to_csv(f"../../data/X{nom}_test_commun_5.csv", index=False)

# Sauvegarde des cibles communes
y3_train.to_csv("../../data/y_train_5modeles.csv", index=False)
y3_test.to_csv("../../data/y_test_5modeles.csv", index=False)

print("Sauvegarde terminée.")


Sauvegarde terminée.


In [44]:

# Création des datasets RF15 et RF18
def construire_dataset_lags(nombre_lags):

    df_lag = df_work.copy()

    # Cible : latence de la prochaine observation
    df_lag["rtt_future"] = df_lag["rtt_avg_ms"].shift(-1)
    df_lag["gap_future_s"] = df_lag["gap_before_s"].shift(-1)

    # Création des variables historiques
    variables = ["rtt_avg_ms", "jitter_ms", "packet_loss_pct"]

    for variable in variables:
        for lag in range(1, nombre_lags + 1):
            df_lag[f"{variable}_lag{lag}"] = (
                df_lag[variable].shift(lag)
            )

    # Vérifier la régularité des intervalles historiques
    condition_intervalles = df_lag["gap_before_s"].between(25, 45)

    for lag in range(1, nombre_lags):
        condition_intervalles &= (
            df_lag["gap_before_s"].shift(lag).between(25, 45)
        )

    # Vérifier la disponibilité des mesures actuelles et futures
    condition_rtt = (
        df_lag["rtt_avg_ms"].notna()
        & df_lag["rtt_future"].notna()
    )

    condition_intervalle_futur = (
        df_lag["gap_future_s"].between(25, 45)
    )

    # Vérifier que toutes les variables historiques existent
    features_historiques = [
        f"{variable}_lag{lag}"
        for variable in variables
        for lag in range(1, nombre_lags + 1)
    ]

    condition_historique = (
        df_lag[features_historiques].notna().all(axis=1)
    )

    # Filtrage final
    masque_valide = (
        condition_intervalles
        & condition_rtt
        & condition_intervalle_futur
        & condition_historique
    )

    df_lag_clean = df_lag.loc[masque_valide].copy()

    # Construction de X et y
    features = features_actuelles + features_historiques

    X = df_lag_clean[features].copy()
    y = df_lag_clean["rtt_future"].copy()

    return df_lag_clean, X, y, features


# Construction RF15
df_lag15_clean, X_lag15, y_lag15, features_lag15 = (
    construire_dataset_lags(15)
)

# Construction RF18
df_lag18_clean, X_lag18, y_lag18, features_lag18 = (
    construire_dataset_lags(18)
)

# Vérification
print("===== RF15 =====")
print("Dimensions :", X_lag15.shape)
print("Cible :", y_lag15.shape)
print("Variables :", len(features_lag15))
print("Valeurs manquantes :", X_lag15.isna().sum().sum())

print("\n===== RF18 =====")
print("Dimensions :", X_lag18.shape)
print("Cible :", y_lag18.shape)
print("Variables :", len(features_lag18))
print("Valeurs manquantes :", X_lag18.isna().sum().sum())


===== RF15 =====
Dimensions : (45610, 51)
Cible : (45610,)
Variables : 51
Valeurs manquantes : 0

===== RF18 =====
Dimensions : (44004, 60)
Cible : (44004,)
Variables : 60
Valeurs manquantes : 0


In [45]:

# Intersection des observations valides
indices_communs_7 = (
    df_lag_clean.index
    .intersection(df_lag5_clean.index)
    .intersection(df_lag8_clean.index)
    .intersection(df_lag9_clean.index)
    .intersection(df_lag12_clean.index)
    .intersection(df_lag15_clean.index)
    .intersection(df_lag18_clean.index)
    .sort_values()
)

print("Observations communes :", len(indices_communs_7))

# Construction des sept datasets
X3_commun = df_lag_clean.loc[
    indices_communs_7, features_lag3_completes
].copy()

X5_commun = df_lag5_clean.loc[
    indices_communs_7, features_lag5_completes
].copy()

X8_commun = df_lag8_clean.loc[
    indices_communs_7, features_lag8
].copy()

X9_commun = df_lag9_clean.loc[
    indices_communs_7, features_lag9
].copy()

X12_commun = df_lag12_clean.loc[
    indices_communs_7, features_lag12
].copy()

X15_commun = df_lag15_clean.loc[
    indices_communs_7, features_lag15
].copy()

X18_commun = df_lag18_clean.loc[
    indices_communs_7, features_lag18
].copy()

# Cible commune
y_commun_7 = df_lag18_clean.loc[
    indices_communs_7, "rtt_future"
].copy()

# Vérification des dimensions
print("\nDimensions :")
for nom, X in [
    ("RF3", X3_commun),
    ("RF5", X5_commun),
    ("RF8", X8_commun),
    ("RF9", X9_commun),
    ("RF12", X12_commun),
    ("RF15", X15_commun),
    ("RF18", X18_commun)
]:
    print(nom, ":", X.shape)

print("y :", y_commun_7.shape)

# Vérification de l'identité des cibles
print(
    "\nCibles identiques :",
    y_commun_7.equals(
        df_lag_clean.loc[indices_communs_7, "rtt_future"]
    )
)


Observations communes : 44004

Dimensions :
RF3 : (44004, 15)
RF5 : (44004, 21)
RF8 : (44004, 30)
RF9 : (44004, 33)
RF12 : (44004, 42)
RF15 : (44004, 51)
RF18 : (44004, 60)
y : (44004,)

Cibles identiques : True


In [46]:

# Découpage commun
split_index_7 = int(len(X18_commun) * 0.8)

def decouper_dataset_7(X, y, split_index):
    X_train = X.iloc[:split_index].copy()
    X_test = X.iloc[split_index:].copy()

    y_train = y.iloc[:split_index].copy()
    y_test = y.iloc[split_index:].copy()

    # Suppression de la dernière ligne train
    X_train = X_train.iloc[:-1]
    y_train = y_train.iloc[:-1]

    return X_train, X_test, y_train, y_test


# Découpage des sept modèles
datasets = {
    "3": X3_commun,
    "5": X5_commun,
    "8": X8_commun,
    "9": X9_commun,
    "12": X12_commun,
    "15": X15_commun,
    "18": X18_commun
}

resultats_decoupage = {}

for nom, X in datasets.items():
    resultats_decoupage[nom] = decouper_dataset_7(
        X, y_commun_7, split_index_7
    )

    Xtr, Xte, ytr, yte = resultats_decoupage[nom]

    print(f"\nRF{nom}")
    print("Train :", Xtr.shape)
    print("Test  :", Xte.shape)

# Vérification des cibles
y_train_commun_7 = resultats_decoupage["3"][2]
y_test_commun_7 = resultats_decoupage["3"][3]

print("\nVérification des cibles :")

for nom in datasets:
    ytr = resultats_decoupage[nom][2]
    yte = resultats_decoupage[nom][3]

    print(
        f"RF{nom} :",
        ytr.equals(y_train_commun_7),
        yte.equals(y_test_commun_7)
    )



RF3
Train : (35202, 15)
Test  : (8801, 15)

RF5
Train : (35202, 21)
Test  : (8801, 21)

RF8
Train : (35202, 30)
Test  : (8801, 30)

RF9
Train : (35202, 33)
Test  : (8801, 33)

RF12
Train : (35202, 42)
Test  : (8801, 42)

RF15
Train : (35202, 51)
Test  : (8801, 51)

RF18
Train : (35202, 60)
Test  : (8801, 60)

Vérification des cibles :
RF3 : True True
RF5 : True True
RF8 : True True
RF9 : True True
RF12 : True True
RF15 : True True
RF18 : True True


In [48]:

for nom, (Xtr, Xte, ytr, yte) in resultats_decoupage.items():

    Xtr.to_csv(
        f"../../data/X{nom}_train_commun_7.csv",
        index=False
    )

    Xte.to_csv(
        f"../../data/X{nom}_test_commun_7.csv",
        index=False
    )

y_train_commun_7.to_csv(
    "../../data/y_train_7modeles.csv",
    index=False
)

y_test_commun_7.to_csv(
    "../../data/y_test_7modeles.csv",
    index=False
)

print("Sauvegarde des sept datasets terminée.")


Sauvegarde des sept datasets terminée.
